# Exercise 2 — Fix yesterday's orders

⏱️ 25 minutes hands-on, then we discuss as a group.

## The situation

Marc, who runs Ops, catches you before standup:

> Three orders from yesterday went out at the wrong price — someone keyed the amount in
> without the decimal point, so we've got a €23,000 order for a pair of boots. And there are
> two QA test rows in there from Friday's release.
>
> Finance exports at 18:00 and whatever's in the files at that point is what they'll book.
> Can you get them corrected before then?

Yesterday's drop is three Parquet files. The cell below copies them into your own storage
container, exactly as they arrived.

In [ ]:
from northtrail import get_spark, path, local_data, where_am_i

print(where_am_i())
spark = get_spark("exercise-2")

ORDERS = path("raw", "orders")

# Yesterday's drop, landed in your container as three files.
(spark.read.parquet(local_data("lake", "raw", "orders"))
      .repartition(3)
      .write.mode("overwrite").parquet(ORDERS))

print(f"{spark.read.parquet(ORDERS).count()} orders at {ORDERS}")

In [ ]:
# Find the rows Marc means. NorthTrail's most expensive product is a €795 pack.
orders = spark.read.parquet(ORDERS)

orders.orderBy(orders.amount.desc()).show(5)
orders.filter("customer_id = 'TEST'").show()

Three orders priced 100x too high, and two rows from QA. Five rows to deal with, out of 402.

Go ahead and fix them.

In [ ]:
# Attempt: correct the three prices.
#TODO: Replace the ??? with appropriate values for the update query.
try:
    spark.sql(f"UPDATE parquet.`{???}` SET amount = round(amount / 100, 2) WHERE amount > ???")
    print("updated")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

## 🔍 What just happened?

_Spark refused the `UPDATE`. Read the message: is this a permissions problem, a typo, or
something Spark will never do against a folder of Parquet files? What is it about a
Parquet file that would make this hard?_

Fine. You have the data and you have Spark. Do it the direct way: read the folder,
correct the rows in memory, and write the result back over the top.

In [ ]:
# Attempt, take two: read it, fix it, write it back where it came from.
from pyspark.sql.functions import col, round as sround, when

fixed = (spark.read.parquet(ORDERS)
              .withColumn("amount", when(col("amount") > 1000, sround(col("amount") / 100, 2))
                                    .otherwise(col("amount")))
              .filter("customer_id <> 'TEST'"))

try:
    fixed.write.mode("overwrite").parquet(ORDERS)
    print("write finished")
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:250]}")

That threw a wall of red. Before reading it closely — check what is actually in the folder now.

In [ ]:
try:
    print(f"{spark.read.parquet(ORDERS).count()} orders")
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:160]}")

## 🔍 What just happened?

_Check that count again — how many of the 402 orders are left? Work out the order of
events inside `mode("overwrite")`: what did Spark delete, and when did it start reading?
Then think about timing. Even in the version where this succeeds, what would Finance's
18:00 export have picked up if it ran mid-write? And when Marc asks what you changed,
what can you show him?_

Luckily the upstream drop still exists, so you can recover.

In [ ]:
# Copy the drop in again from upstream.
(spark.read.parquet(local_data("lake", "raw", "orders"))
      .repartition(3)
      .write.mode("overwrite").parquet(ORDERS))

print(f"recovered: {spark.read.parquet(ORDERS).count()} orders")

## 💡 Concept: file format vs table format

Parquet is a **file format**. It defines how columns and rows are laid out as bytes inside
one file, and it is very good at that. It says nothing about the *folder* — nothing about
which files belong together, what happens when two writers touch it at once, or how a reader
gets a consistent view while a write is in progress. There is no such thing as changing a
row: a file is written once and never modified.

A **table format** adds that missing layer. It keeps a log next to the data files recording
which files make up the table right now, and every change appends a new entry to that log.
A reader follows the log, so it always sees one complete version, never a half-finished
write. **Delta Lake** is one such format, and Spark can add it to a folder of Parquet files
you already have — the files stay exactly where they are.

In [ ]:
# The data files don't move or get rewritten. This only adds the log.
#TODO: Replace the ??? with appropriate values for the update and select queries.
spark.sql(f"CONVERT TO DELTA parquet.`{ORDERS}`")

# Now the fixes Spark refused to do earlier.
spark.sql(f"??? delta.`{ORDERS}` SET amount = round(amount / 100, 2) WHERE amount > 1000")
spark.sql(f"DELETE FROM delta.`{ORDERS}` WHERE customer_id = 'TEST'")

spark.sql(f'''
    SELECT count(*) AS orders, round(max(amount), 2) AS largest, round(sum(amount), 2) AS total
    FROM ???.`{ORDERS}`
''').show()

In [ ]:
# Marc's other question: what exactly did you change?
spark.sql(f"DESCRIBE HISTORY delta.`{ORDERS}`").select("version", "timestamp", "operation").show(truncate=False)

400 orders, largest is €795, and there's a record of every change you made.

Later that week a colleague picks up a related ticket and messages you:

> What tables do we actually have in there? What columns are on the orders one, and is it
> current or is it from last month?

In [ ]:
# What can you tell them?
spark.sql("SHOW TABLES").show()

print(f"The only answer you have is the path itself:\n  {ORDERS}")

## 🔍 What just happened?

_Nothing lists the table you just fixed. How does a colleague find it, and how do they
learn its columns? Now multiply your answer by forty tables and six people._

## 💡 Concept: the catalog

A **catalog** is the layer that maps a *name* to a table. It holds the names, schemas,
locations and ownership, so people query `orders` instead of remembering a URL, and can list
what exists without reading any data. The three layers stack: file format (bytes in a file),
table format (which files are the table), catalog (what the tables are called and who may
read them).

⚠️ **The catalog you're about to use is a stand-in.** Spark has a small built-in one that
lives in this notebook session and disappears when it stops. It is enough to see what a
catalog does, and it is not what you'd use in production — there you'd point Spark at Unity
Catalog, AWS Glue, or an Iceberg REST catalog, shared across every engine and team, with
access control attached.

In [ ]:
# Register the table under a name. The data doesn't move -- this records where it lives.
spark.sql("DROP TABLE IF EXISTS orders")
spark.sql(f"CREATE TABLE orders USING DELTA LOCATION '{ORDERS}'")

spark.sql("SHOW TABLES").show()
spark.sql("DESCRIBE orders").show()

In [ ]:
# Your colleague can now answer all three questions without knowing any paths.
#TODO: Replace the ??? with appropriate values for the DESCRIBE DETAIL query.
spark.sql("SELECT region, round(sum(amount), 2) AS revenue FROM orders GROUP BY region ORDER BY region").show()

spark.sql("DESCRIBE DETAIL ???").select("format", "numFiles", "sizeInBytes", "lastModified").show(truncate=False)

## Debrief

- The overwrite destroyed 402 orders and Spark reported it as a read error. What would have
  to be true about your setup for that to be caught before Finance noticed?
- File format, table format, catalog are three separate layers here, and a warehouse bundles
  all three into one product. What do you actually gain from them being separable — and what
  does keeping them separate cost you?
- You now have a log of every change to this table. Name something that log makes possible
  that you could not do with plain Parquet files, beyond undoing mistakes.